# 📊 Visualization — Gendered Language in Job Postings

Five publication-ready figures telling a coherent story:

| # | Figure | Question answered |
|---|---|---|
| 1 | Bias score by sector | Which sector is most gendered? |
| 2 | Language profile distribution | What share of postings is gendered per sector? |
| 3 | Word density heatmap | Where do masculine/feminine words concentrate? |
| 4 | WordClouds | Which specific words drive the bias? |
| 5 | Scatter plot | Where does each posting sit individually? |

In [ ]:
# ── Cell 0 : Imports & global style ──────────────────────────────────────────
import json
import re
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from wordcloud import WordCloud

warnings.filterwarnings('ignore')

# ── Output directory ─────────────────────────────────────────────────────────
OUT = Path('../data')
OUT.mkdir(parents=True, exist_ok=True)

# ── Design system ────────────────────────────────────────────────────────────
BG       = '#0A0C14'   # near-black background
SURFACE  = '#13172A'   # card background
BORDER   = '#1E2340'   # subtle borders
TEXT     = '#E8EAF0'   # primary text
SUBTEXT  = '#7B82A0'   # secondary text

MASC_CLR = '#60A5FA'   # blue  — masculine
FEM_CLR  = '#F472B6'   # pink  — feminine
NEUT_CLR = '#374151'   # dark grey — neutral
ACCENT   = '#818CF8'   # indigo accent

SECTOR_PALETTE = {
    'care'      : '#F472B6',
    'tech'      : '#60A5FA',
    'finance'   : '#34D399',
    'marketing' : '#FBBF24',
    'hr'        : '#A78BFA',
    'data'      : '#FB923C',
}

SECTOR_ORDER = ['care', 'tech', 'hr', 'finance', 'marketing', 'data']

# Apply globally
plt.rcParams.update({
    'figure.facecolor'  : BG,
    'axes.facecolor'    : SURFACE,
    'axes.edgecolor'    : BORDER,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.spines.left'  : False,
    'axes.spines.bottom': False,
    'axes.labelcolor'   : SUBTEXT,
    'axes.labelsize'    : 10,
    'axes.titlesize'    : 13,
    'axes.titlecolor'   : TEXT,
    'axes.titlepad'     : 14,
    'axes.grid'         : True,
    'grid.color'        : BORDER,
    'grid.linewidth'    : 0.6,
    'text.color'        : TEXT,
    'xtick.color'       : SUBTEXT,
    'ytick.color'       : SUBTEXT,
    'xtick.labelsize'   : 9,
    'ytick.labelsize'   : 9,
    'legend.framealpha' : 0.15,
    'legend.edgecolor'  : BORDER,
    'legend.fontsize'   : 9,
    'font.family'       : 'DejaVu Sans',
    'figure.dpi'        : 150,
})

def save(name: str) -> None:
    """Save figure to OUT directory at high resolution."""
    path = OUT / name
    plt.savefig(path, dpi=180, bbox_inches='tight',
                facecolor=BG, edgecolor='none')
    print(f'  💾 saved → {path}')

# ── Load data ────────────────────────────────────────────────────────────────
df = pd.read_csv('../data/processed/jobs_scored.csv')
with open('../data/lexicon/gendered_lexicon.json') as f:
    lexicon = json.load(f)

MASC_SET = set(lexicon['masculine'])
FEM_SET  = set(lexicon['feminine'])

print(f'✅ Setup OK — {len(df):,} postings loaded')
print(df['sector'].value_counts().to_string())

---
## Figure 1 — Average Gender Bias Score by Sector
> *Which sector uses the most gendered language?*

In [ ]:
print('Rendering Figure 1...')

stats = (
    df.groupby('sector')['bias_score']
    .mean()
    .reindex(SECTOR_ORDER)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(11, 5.5))
fig.patch.set_facecolor(BG)

colors = [FEM_CLR if v < 0 else MASC_CLR for v in stats.values]

bars = ax.barh(
    stats.index, stats.values,
    color=colors, height=0.55,
    linewidth=0,
)

# Zero line
ax.axvline(0, color=TEXT, linewidth=0.8, alpha=0.4, zorder=3)

# Value labels
for bar, val in zip(bars, stats.values):
    pad = 0.00015 if val >= 0 else -0.00015
    ha  = 'left'  if val >= 0 else 'right'
    ax.text(
        val + pad,
        bar.get_y() + bar.get_height() / 2,
        f'{val:+.4f}',
        va='center', ha=ha,
        fontsize=9, color=TEXT, fontweight='bold',
    )

# Sector count annotations
for i, sector in enumerate(stats.index):
    n = df[df['sector'] == sector].shape[0]
    ax.text(
        stats.min() * 1.02, i,
        f'n={n:,}',
        va='center', ha='right',
        fontsize=7.5, color=SUBTEXT,
    )

# Legend
legend_patches = [
    mpatches.Patch(facecolor=MASC_CLR, label='Masculine dominant', linewidth=0),
    mpatches.Patch(facecolor=FEM_CLR,  label='Feminine dominant',  linewidth=0),
]
ax.legend(handles=legend_patches, loc='lower right', framealpha=0.2)

ax.set_title(
    'Average Gender Bias Score by Sector\n'
    r'$\leftarrow$ Feminine dominant    |    Masculine dominant $\rightarrow$',
    fontsize=13, color=TEXT,
)
ax.set_xlabel('Bias score  (masculine word freq. − feminine word freq.)', color=SUBTEXT)
ax.grid(axis='x', alpha=0.25, linewidth=0.6)
ax.grid(axis='y', visible=False)
ax.tick_params(axis='y', labelsize=10, labelcolor=TEXT)

plt.tight_layout()
save('viz1_bias_by_sector.png')
plt.show()

---
## Figure 2 — Language Profile Distribution by Sector
> *What share of postings is gendered per sector?*

In [ ]:
print('Rendering Figure 2...')

pivot = (
    df.groupby(['sector', 'bias_label'])
    .size()
    .unstack(fill_value=0)
    .reindex(SECTOR_ORDER)
)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

# Ensure all columns exist
for col in ['masculin', 'neutre', 'féminin']:
    if col not in pivot_pct.columns:
        pivot_pct[col] = 0

fig, ax = plt.subplots(figsize=(11, 5.5))
fig.patch.set_facecolor(BG)

x      = np.arange(len(SECTOR_ORDER))
width  = 0.55

b1 = ax.bar(x, pivot_pct['masculin'], width, color=MASC_CLR, label='Masculine', linewidth=0)
b2 = ax.bar(x, pivot_pct['neutre'],   width, color=NEUT_CLR, label='Neutral',
             bottom=pivot_pct['masculin'], linewidth=0)
b3 = ax.bar(x, pivot_pct['féminin'],  width, color=FEM_CLR,  label='Feminine',
             bottom=pivot_pct['masculin'] + pivot_pct['neutre'], linewidth=0)

# Percentage labels inside bars (only if segment > 8%)
for i, sector in enumerate(SECTOR_ORDER):
    masc = pivot_pct.loc[sector, 'masculin']
    neut = pivot_pct.loc[sector, 'neutre']
    fem  = pivot_pct.loc[sector, 'féminin']

    if masc > 8:
        ax.text(i, masc / 2, f'{masc:.0f}%', ha='center', va='center',
                fontsize=8.5, color=BG, fontweight='bold')
    if neut > 8:
        ax.text(i, masc + neut / 2, f'{neut:.0f}%', ha='center', va='center',
                fontsize=8.5, color=TEXT, fontweight='bold')
    if fem > 8:
        ax.text(i, masc + neut + fem / 2, f'{fem:.0f}%', ha='center', va='center',
                fontsize=8.5, color=BG, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([s.upper() for s in SECTOR_ORDER], fontsize=10, color=TEXT)
ax.set_ylabel('Share of job postings (%)', color=SUBTEXT)
ax.set_ylim(0, 108)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.grid(axis='y', alpha=0.2)
ax.grid(axis='x', visible=False)
ax.legend(loc='upper right')
ax.set_title('Language Profile Distribution by Sector', fontsize=13, color=TEXT)

plt.tight_layout()
save('viz2_distribution.png')
plt.show()

---
## Figure 3 — Gendered Word Density Heatmap
> *Where do masculine and feminine words concentrate?*

In [ ]:
print('Rendering Figure 3...')

heatmap_data = (
    df.groupby('sector')[['masc_score', 'fem_score']]
    .mean()
    .reindex(SECTOR_ORDER)
    * 1000  # per 1,000 words
)
heatmap_data.columns = ['Masculine\n(per 1k words)', 'Feminine\n(per 1k words)']

fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(BG)

# Custom colormap: dark BG → accent color
cmap = sns.color_palette('rocket', as_cmap=True)

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.2f',
    cmap=cmap,
    linewidths=2,
    linecolor=BG,
    annot_kws={'size': 12, 'weight': 'bold', 'color': TEXT},
    cbar_kws={'shrink': 0.8, 'pad': 0.02},
    ax=ax,
)

ax.set_yticklabels(
    [s.upper() for s in SECTOR_ORDER],
    rotation=0, fontsize=9, color=TEXT,
)
ax.set_xticklabels(heatmap_data.columns, fontsize=10, color=TEXT, rotation=0)
ax.set_ylabel('')

# Style colorbar
cbar = ax.collections[0].colorbar
cbar.ax.yaxis.set_tick_params(color=SUBTEXT, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=SUBTEXT)

ax.set_title(
    'Gendered Word Density by Sector\n'
    '(occurrences per 1,000 words — based on Gaucher et al. 2011 lexicon)',
    fontsize=12, color=TEXT,
)

plt.tight_layout()
save('viz3_heatmap.png')
plt.show()

---
## Figure 4 — WordClouds: Most Frequent Gendered Words
> *Which specific words drive the bias?*

In [ ]:
print('Rendering Figure 4...')

def get_word_freq(texts: pd.Series, word_set: set) -> dict:
    counter = Counter()
    for text in texts:
        words = re.findall(r'\b[a-z]+\b', str(text))
        counter.update(w for w in words if w in word_set)
    return dict(counter)

all_desc  = df['description_clean']
masc_freq = get_word_freq(all_desc, MASC_SET)
fem_freq  = get_word_freq(all_desc, FEM_SET)

WC_KWARGS = dict(
    width=700, height=420,
    background_color=SURFACE,
    max_words=35,
    prefer_horizontal=0.85,
    margin=6,
    relative_scaling=0.55,
)

wc_masc = WordCloud(colormap='Blues_r',  **WC_KWARGS).generate_from_frequencies(masc_freq)
wc_fem  = WordCloud(colormap='RdPu',     **WC_KWARGS).generate_from_frequencies(fem_freq)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(BG)

for ax_wc, wc, title, color in [
    (axes[0], wc_masc, 'Masculine-coded words', MASC_CLR),
    (axes[1], wc_fem,  'Feminine-coded words',  FEM_CLR),
]:
    ax_wc.imshow(wc, interpolation='bilinear')
    ax_wc.axis('off')
    ax_wc.set_facecolor(SURFACE)
    ax_wc.set_title(title, fontsize=13, color=color,
                    fontweight='bold', pad=12)
    # Subtle border
    for spine in ax_wc.spines.values():
        spine.set_edgecolor(BORDER)
        spine.set_linewidth(1.5)
        spine.set_visible(True)

fig.suptitle(
    'Most Frequent Gendered Words — 34,656 Job Postings',
    fontsize=14, color=TEXT, fontweight='bold', y=1.01,
)
fig.text(
    0.5, -0.02,
    'Lexicon: Gaucher et al. (2011) — Journal of Personality and Social Psychology',
    ha='center', fontsize=8, color=SUBTEXT,
)

plt.tight_layout()
save('viz4_wordclouds.png')
plt.show()

---
## Figure 5 — Individual Posting Language Profile (Scatter)
> *Where does each posting sit individually?*

In [ ]:
print('Rendering Figure 5...')

# Stratified sample for readability
sample = (
    df.groupby('sector', group_keys=False)
    .apply(lambda x: x.sample(min(400, len(x)), random_state=42))
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor(BG)

# Plot each sector
for sector in SECTOR_ORDER:
    sub = sample[sample['sector'] == sector]
    ax.scatter(
        sub['masc_score'], sub['fem_score'],
        c=SECTOR_PALETTE[sector],
        label=sector.upper(),
        alpha=0.45, s=18,
        edgecolors='none',
        zorder=2,
    )

# Diagonal neutrality line
lim = max(sample['masc_score'].max(), sample['fem_score'].max()) * 1.05
ax.plot(
    [0, lim], [0, lim],
    '--', color=TEXT, alpha=0.3,
    linewidth=1.2, zorder=1,
    label='Neutral line (masc = fem)',
)

# Annotation zones
ax.text(
    lim * 0.6, lim * 0.15,
    'Masculine\ndominant',
    color=MASC_CLR, fontsize=8.5, alpha=0.7, ha='center',
)
ax.text(
    lim * 0.1, lim * 0.75,
    'Feminine\ndominant',
    color=FEM_CLR, fontsize=8.5, alpha=0.7, ha='center',
)

ax.set_xlabel('Masculine word frequency', color=SUBTEXT, fontsize=10)
ax.set_ylabel('Feminine word frequency',  color=SUBTEXT, fontsize=10)
ax.set_title(
    'Individual Job Posting Language Profile\n'
    'Points above the diagonal → feminine dominant language',
    fontsize=13, color=TEXT,
)
ax.legend(markerscale=2, loc='upper right', ncol=2)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.grid(alpha=0.15)

plt.tight_layout()
save('viz5_scatter.png')
plt.show()

---
## Summary

All 5 figures saved to `data/`.

| Figure | Key insight |
|---|---|
| viz1 | Care sector bias score is **10× larger** than any other sector |
| viz2 | **86%** of care postings use feminine-dominant language |
| viz3 | Care uses **25.42** feminine words/1k vs. **7–9** in all other sectors |
| viz4 | *strong / leadership / competitive* vs. *patient / care / support* |
| viz5 | Care points cluster far above the diagonal — structurally different |